---
## Task 1 — Possession Advantage of Winning Teams

### Analytic question formulation
Do **winning teams** hold significantly greater **ball possession** than the team they beat?
For each decisive match we compute the winning team's possession share minus the losing
team's possession share, then test whether this **possession difference** is greater than
zero on average.

**H0:** mean(possession difference) = 0 (no possession advantage for winners)
**H1:** mean(possession difference) > 0 (winners hold more possession)

### Data wrangling
Load the raw match-level CSV, drop drawn matches (there is no "winner" to compare), and
derive a single `possession_difference` column: the winning team's possession share minus
the losing team's possession share.

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import scipy.stats as st

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your CSV file
# file_path = '/content/drive/MyDrive/Colab/world_cup_match_data.csv'

file_path = "../../data/world_cup_match_data.csv"

df = pd.read_csv(file_path)
print("CSV imported successfully!")
print(df.head())

Mounted at /content/drive
CSV imported successfully!
    timestamp              date_GMT    status  attendance home_team_name  \
0  1781204400  Jun 11 2026 - 7:00pm  complete       80824         Mexico   
1  1781229600  Jun 12 2026 - 2:00am  complete       44985    South Korea   
2  1781290800  Jun 12 2026 - 7:00pm  complete       43002         Canada   
3  1781312400  Jun 13 2026 - 1:00am  complete       70492          USMNT   
4  1781377200  Jun 13 2026 - 7:00pm  complete       67966          Qatar   

           away_team_name  referee  Game Week  Pre-Match PPG (Home)  \
0            South Africa      NaN        1.0                   0.0   
1          Czech Republic      NaN        1.0                   0.0   
2  Bosnia and Herzegovina      NaN        1.0                   0.0   
3                Paraguay      NaN        1.0                   0.0   
4             Switzerland      NaN        1.0                   0.0   

   Pre-Match PPG (Away)  ...  odds_ft_home_team_win  odds_ft_dr

In [2]:
df1 = df[
        [
            'home_team_name',
            'away_team_name',
            'home_team_goal_count',
            'away_team_goal_count',
            'home_team_possession',
            'away_team_possession'
        ]
    ].copy()

# Determine winning team
# 0 = Home team wins, 1 = Away team wins, -1 = Draw
df1['winning_team'] = df1.apply(
    lambda row: 0 if row['home_team_goal_count'] > row['away_team_goal_count']
    else 1 if row['away_team_goal_count'] > row['home_team_goal_count']
    else -1,
    axis=1
)
df1 = df1[df1['winning_team'] != -1].copy()

df1['possession_difference'] = df1.apply(
    lambda row: (
        row['home_team_possession'] - row['away_team_possession']
        if row['winning_team'] == 0
        else row['away_team_possession'] - row['home_team_possession']
    ),
    axis=1
)

df1.reset_index(drop=True, inplace=True)
print("Population size (decisive matches):", len(df1))
df1.head()

Population size (decisive matches): 80


,home_team_name,away_team_name,home_team_goal_count,away_team_goal_count,home_team_possession,away_team_possession,winning_team,possession_difference
0,Mexico,South Africa,2,0,61,39,0,22
1,South Korea,Czech Republic,2,1,62,38,0,24
2,USMNT,Paraguay,4,1,65,35,0,30
3,Haiti,Scotland,0,1,54,46,1,-8
4,Australia,Turkey,2,0,28,72,0,-44


### Data preparation and sampling
**Population:** one `possession_difference` value per decisive match.
**Sample:** a **simple random sample of n = 30** possession-difference values, used for the
one-sample test against a hypothesised mean of 0.

In [3]:
n = 30
poss_diff_sample = df1['possession_difference'].sample(n=n, random_state=7).reset_index(drop=True)

print("Sample n:", len(poss_diff_sample))

Sample n: 30


### Descriptive statistics

In [4]:
poss_diff_sample.describe()

,possession_difference
count,30.000000
mean,12.800000
std,22.740515
min,-58.000000
25%,4.000000
50%,15.000000
75%,28.000000
max,46.000000


### Inferential statistics — Confidence interval (95%)

In [5]:
m, sem = poss_diff_sample.mean(), st.sem(poss_diff_sample)
ci = st.t.interval(0.95, df=len(poss_diff_sample) - 1, loc=m, scale=sem)
print(f"Possession difference: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

Possession difference: mean = 12.800, 95% CI = (4.309, 21.291)


### Inferential statistics — One-Sample t-Test (one-tailed)
H0: μ(possession difference) = 0  vs.  H1: μ(possession difference) > 0

In [6]:
t_stat, p_val = st.ttest_1samp(poss_diff_sample, 0, alternative='greater')
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")

alpha = 0.05
conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
print("Conclusion:", conclusion, "at the 5% significance level.")

t-statistic = 3.083, p-value = 0.0022
Conclusion: Reject H0 at the 5% significance level.


In [7]:
# sig = "statistically significant" if p_val < alpha else "not statistically significant"
# print(
#     f"Interpretation: In the sample, winning teams held {m:.2f} percentage points more "
#     f"possession than the teams they beat, on average. With p = {p_val:.4f}, this result is "
#     f"{sig} at the 5% level, so we {conclusion.lower()} that winning teams hold significantly "
#     f"more possession than losing teams."
# )

Interpretation: In the sample, winning teams held 12.80 percentage points more possession than the teams they beat, on average. With p = 0.0022, this result is statistically significant at the 5% level, so we reject h0 that winning teams hold significantly more possession than losing teams.


**interpretation:** winning teams hold significantly more possession than losing teams.